# Phase 8 — Contrôle

La volatilité retrouve la phase 5. La VaR est le rendement du rang historique. L'expected shortfall est la moyenne de cette queue. La corrélation moyenne est celle des paires hors diagonale.

In [1]:
import math
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
for candidate in (PROJECT_ROOT, *PROJECT_ROOT.parents):
    if (candidate / "src").is_dir():
        PROJECT_ROOT = candidate
        break
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import numpy as np
import pandas as pd

from src.config import MONTHS_PER_YEAR
from src.portfolio_construction.build_portfolios import METRICS_FILE, STRATEGY_IDS
from src.risk.build_risk import VAR_LEVEL, build_risk
from src.risk.dashboard import DASHBOARD_FILE, SECTION_TITLES

written = build_risk()
metrics = pd.read_csv(written["metrics"])
tail = pd.read_csv(written["tail"])
summary = pd.read_csv(written["summary"])
matrix = pd.read_csv(written["matrix"])
published = pd.read_csv(METRICS_FILE)

# Three books, then the fund: the fund row is published for the comparison.
assert list(metrics["strategy"]) == list(STRATEGY_IDS) + ["benchmark"]
assert metrics["months"].nunique() == 1
assert metrics["window_start"].nunique() == 1
assert metrics["window_end"].nunique() == 1
count = int(metrics["months"].iloc[0])
rank = math.ceil(VAR_LEVEL * count)
assert len(summary) == 1
assert int(summary["months"].iloc[0]) == count
symbols = matrix["symbol"].astype(str).tolist()
assert symbols == list(matrix.columns[1:])
assert int(summary["names"].iloc[0]) == len(symbols)
square = matrix.set_index("symbol").loc[symbols, symbols]
assert np.allclose(np.diag(square.to_numpy(dtype=float)), 1.0)
pairs = square.to_numpy(dtype=float)[np.triu_indices(len(symbols), k=1)]
assert abs(float(pairs.mean()) - float(summary["average_correlation"].iloc[0])) <= 1e-12

for strategy in list(STRATEGY_IDS) + ["benchmark"]:
    row = metrics.loc[metrics["strategy"].eq(strategy)].iloc[0]
    if strategy != "benchmark":
        official = published.loc[published["strategy"].eq(strategy)].iloc[0]
        assert abs(float(row["annualized_volatility"]) - float(official["annualized_volatility"])) <= 1e-12
    book = tail.loc[tail["strategy"].eq(strategy)].sort_values("rank")
    assert book["rank"].astype(int).tolist() == list(range(1, rank + 1))
    returns = book["portfolio_return"].astype(float)
    assert abs(float(returns.iloc[-1]) - float(row["var_95"])) <= 1e-12
    assert abs(float(returns.mean()) - float(row["expected_shortfall_95"])) <= 1e-12
    assert float(row["expected_shortfall_95"]) <= float(row["var_95"]) + 1e-12

html = DASHBOARD_FILE.read_text(encoding="utf-8")
assert written["dashboard"] == DASHBOARD_FILE
for title in SECTION_TITLES:
    word = title.split()[0]
    escaped = "".join(
        f"\\u{ord(character):04x}" if ord(character) > 127 else character
        for character in word
    )
    assert word in html or escaped in html
assert str(metrics["window_end"].iloc[0]) in html
print(f"Période : {metrics['window_start'].iloc[0]} – {metrics['window_end'].iloc[0]}. Mois : {count}.")
print("Résultat : PASS")


Période : 2016-11 – 2026-10. Mois : 120.
Résultat : PASS
